Semantic Chunking

In [ ]:
import sqlite3
import numpy as np
import re
import os
from sentence_transformers import SentenceTransformer

In [ ]:
def ingest_document(file_path):

    # Read the .txt file
    if not isinstance(file_path, str) or not file_path.endswith('.txt'):
        raise ValueError("Input must be a string path to a .txt file.")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        full_text = f.read().strip()
    
    document_name = os.path.basename(file_path)
    
    if not full_text:
        print("Empty file; nothing to process.")
        return
    
    # Step 2: Semantic chunking - First by paragraphs, then by sentences (punctuation)
    # Paragraph split
    para_chunks = re.split(r'\n\n+', full_text)
    para_chunks = [p.strip() for p in para_chunks if p.strip()]
    
    all_chunks = []
    for para in para_chunks:
        # Sentence split within paragraph using regex 
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', para)
        sentences = [s.strip() for s in sentences if s.strip()]
        all_chunks.extend(sentences)
    
    # Generate chunk IDs and  embeddings
    model = SentenceTransformer('all-MiniLM-L6-v2') 
    embedding_dim = 384 
    chunk_data = []
    for i, chunk in enumerate(all_chunks):
        chunk_id = f"{document_name}_chunk_{i+1}"
        
        embedding_array = model.encode(chunk)
        embedding = embedding_array.astype(np.float32).tobytes()
        
        chunk_data.append((document_name, chunk_id, chunk, embedding))
    
    # Store in SQLite
    db_path = 'chunks.db'  
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    
    # Create table
    c.execute('''
    CREATE TABLE IF NOT EXISTS chunks (
        document_name TEXT,
        chunk_id TEXT PRIMARY KEY,
        chunk_text TEXT,
        embedding_vector BLOB
    )
    ''')
    
    c.executemany('INSERT OR REPLACE INTO chunks VALUES (?, ?, ?, ?)', chunk_data)
    conn.commit()
    
    c.execute('SELECT document_name, chunk_id, chunk_text FROM chunks WHERE document_name = ?', (document_name,))
    results = c.fetchall()
    print(f"Ingestion complete for '{document_name}'. Stored {len(results)} chunks:")
    for row in results:
        print(f"  - ID: {row[1]}, Text preview: {row[2][:50]}...")
    
    conn.close()
    return db_path 

In [4]:
sample_file = 'demo.txt'

db = ingest_document(sample_file)

Ingestion complete for 'demo.txt'. Stored 2 chunks:
  - ID: demo.txt_chunk_1, Text preview: Hello, my name is grisa....
  - ID: demo.txt_chunk_2, Text preview: I am AI/ML intern at Cybercom Creation....
